# Lesson 6: Essay Writer

In [1]:
from dotenv import load_dotenv
import json

_ = load_dotenv()

In [2]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List
import operator
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage, ChatMessage

memory = SqliteSaver.from_conn_string(":memory:")

In [ ]:
class AgentState(TypedDict):
    task: str
    plan: str
    draft: str
    critique: str
    content: List[str]
    revision_number: int
    max_revisions: int

In [3]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4.1", temperature=0)

In [ ]:
PLAN_PROMPT = """You are an expert writer tasked with writing a high level outline of an essay. \
Write such an outline for the user provided topic. Give an outline of the essay along with any relevant notes \
or instructions for the sections."""

In [ ]:
WRITER_PROMPT = """You are an essay assistant tasked with writing excellent 5-paragraph essays.\
Generate the best essay possible for the user's request and the initial outline. \
If the user provides critique, respond with a revised version of your previous attempts. \
Utilize all the information below as needed: 

------

{content}"""

In [ ]:
REFLECTION_PROMPT = """You are a teacher grading an essay submission. \
Generate critique and recommendations for the user's submission. \
Provide detailed recommendations, including requests for length, depth, style, etc."""

In [ ]:
RESEARCH_PLAN_PROMPT = """You are a researcher charged with providing information that can \
be used when writing the following essay. Generate a list of search queries that will gather \
any relevant information. Only generate 3 queries max."""


In [ ]:
RESEARCH_CRITIQUE_PROMPT = """You are a researcher charged with providing information that can \
be used when making any requested revisions (as outlined below). \
Generate a list of search queries that will gather any relevant information. Only generate 3 queries max."""


In [ ]:
from langchain_core.pydantic_v1 import BaseModel

class Queries(BaseModel):
    queries: List[str]

In [ ]:
from tavily import TavilyClient
import os
tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

In [ ]:
def plan_node(state: AgentState):
    messages = [
        SystemMessage(content=PLAN_PROMPT), 
        HumanMessage(content=state['task'])
    ]
    response = model.invoke(messages)
    return {"plan": response.content}

In [4]:
trend_scout_prompt = """
【角色】
你是“小红书护肤趋势侦察兵”。

【目标】
在「日常护肤」领域，生成适合在小红书传播的内容选题方向。
选题要真实、可执行、容易被保存与转发。

【硬性约束（必须遵守）】
- 只讨论日常护肤：清洁、保湿、修护、防晒、基础抗老。
- 不涉及疾病治疗、处方药、医美项目、诊断建议。
- 禁止使用或暗示“根治、永久、立刻见效、医学证明”等表述。
- 避免制造焦虑或恐吓式表达。

【输出要求（必须严格JSON数组）】
请输出一个JSON列表，每个选题包含：
- topic: 选题名称
- target_group: 适合人群
- core_pain: 核心护肤痛点
- hook: 反常识 / 对比 / 避坑钩子
- content_form: 内容形态建议（清单 / 避坑 / 教程 / 对比）
- risk_note: 可能的合规风险提示

格式如下:
[
  {"topic_index_1": {"topic": string, "target_group": string, "core_pain": string, "hook": string, "content_form": string, "risk_note": string}},
  {"topic_index_2": {"topic": string, "target_group": string, "core_pain": string, "hook": string, "content_form": string, "risk_note": string}},
  {"topic_index_3": {"topic": string, "target_group": string, "core_pain": string, "hook": string, "content_form": string, "risk_note": string}},
]

"""

In [5]:
print(trend_scout_prompt)


【角色】
你是“小红书护肤趋势侦察兵”。

【目标】
在「日常护肤」领域，生成适合在小红书传播的内容选题方向。
选题要真实、可执行、容易被保存与转发。

【硬性约束（必须遵守）】
- 只讨论日常护肤：清洁、保湿、修护、防晒、基础抗老。
- 不涉及疾病治疗、处方药、医美项目、诊断建议。
- 禁止使用或暗示“根治、永久、立刻见效、医学证明”等表述。
- 避免制造焦虑或恐吓式表达。

【输出要求（必须严格JSON数组）】
请输出一个JSON列表，每个选题包含：
- topic: 选题名称
- target_group: 适合人群
- core_pain: 核心护肤痛点
- hook: 反常识 / 对比 / 避坑钩子
- content_form: 内容形态建议（清单 / 避坑 / 教程 / 对比）
- risk_note: 可能的合规风险提示

格式如下:
[
  {"topic_index_1": {"topic": string, "target_group": string, "core_pain": string, "hook": string, "content_form": string, "risk_note": string}},
  {"topic_index_2": {"topic": string, "target_group": string, "core_pain": string, "hook": string, "content_form": string, "risk_note": string}},
  {"topic_index_3": {"topic": string, "target_group": string, "core_pain": string, "hook": string, "content_form": string, "risk_note": string}},
]




In [6]:
messages = [
    SystemMessage(content=trend_scout_prompt), 
    HumanMessage(content="生成5个选题")
]
response = model.invoke(messages)

In [7]:
print(response.content)

[
  {"topic_index_1": {"topic": "洗脸次数越多越干净？日常清洁的3个误区", "target_group": "油皮、混合皮、学生党", "core_pain": "担心脸洗不干净导致爆痘或毛孔粗大", "hook": "洗脸次数多反而可能伤害皮肤屏障？", "content_form": "避坑+对比", "risk_note": "避免暗示单一清洁方式适合所有人，强调因人而异"}},
  {"topic_index_2": {"topic": "保湿≠厚涂面霜！不同肤质的保湿方案清单", "target_group": "干皮、油皮、换季敏感人群", "core_pain": "保湿做不到位，皮肤总是干燥或出油", "hook": "油皮也需要保湿？保湿不是越厚越好！", "content_form": "清单+教程", "risk_note": "不推荐具体品牌或成分，避免功效夸大"}},
  {"topic_index_3": {"topic": "防晒霜怎么选？物理防晒和化学防晒的区别一看就懂", "target_group": "户外党、通勤族、学生", "core_pain": "不知道如何挑选适合自己的防晒产品", "hook": "防晒不是越高倍数越好？", "content_form": "对比+避坑", "risk_note": "不涉及防晒产品对疾病的预防，仅讨论日常防护"}},
  {"topic_index_4": {"topic": "熬夜后皮肤急救：3步修护小技巧", "target_group": "上班族、学生党、夜猫子", "core_pain": "熬夜后皮肤暗沉、干燥、状态差", "hook": "熬夜后护肤不是越多步骤越好！", "content_form": "教程+清单", "risk_note": "不承诺立竿见影效果，强调日常修护"}},
  {"topic_index_5": {"topic": "抗老要趁早？25+基础抗老护肤流程拆解", "target_group": "25岁以上、初老焦虑人群", "core_pain": "担心细纹、松弛，想入门抗老但无从下手", "hook": "抗老不是越贵越有效，基础护肤也很重要！", "content_form": "清单+教程",

In [8]:
trend_options = json.loads(response.content)

In [9]:
for i in trend_options:
    print(i)

{'topic_index_1': {'topic': '洗脸次数越多越干净？日常清洁的3个误区', 'target_group': '油皮、混合皮、学生党', 'core_pain': '担心脸洗不干净导致爆痘或毛孔粗大', 'hook': '洗脸次数多反而可能伤害皮肤屏障？', 'content_form': '避坑+对比', 'risk_note': '避免暗示单一清洁方式适合所有人，强调因人而异'}}
{'topic_index_2': {'topic': '保湿≠厚涂面霜！不同肤质的保湿方案清单', 'target_group': '干皮、油皮、换季敏感人群', 'core_pain': '保湿做不到位，皮肤总是干燥或出油', 'hook': '油皮也需要保湿？保湿不是越厚越好！', 'content_form': '清单+教程', 'risk_note': '不推荐具体品牌或成分，避免功效夸大'}}
{'topic_index_3': {'topic': '防晒霜怎么选？物理防晒和化学防晒的区别一看就懂', 'target_group': '户外党、通勤族、学生', 'core_pain': '不知道如何挑选适合自己的防晒产品', 'hook': '防晒不是越高倍数越好？', 'content_form': '对比+避坑', 'risk_note': '不涉及防晒产品对疾病的预防，仅讨论日常防护'}}
{'topic_index_4': {'topic': '熬夜后皮肤急救：3步修护小技巧', 'target_group': '上班族、学生党、夜猫子', 'core_pain': '熬夜后皮肤暗沉、干燥、状态差', 'hook': '熬夜后护肤不是越多步骤越好！', 'content_form': '教程+清单', 'risk_note': '不承诺立竿见影效果，强调日常修护'}}
{'topic_index_5': {'topic': '抗老要趁早？25+基础抗老护肤流程拆解', 'target_group': '25岁以上、初老焦虑人群', 'core_pain': '担心细纹、松弛，想入门抗老但无从下手', 'hook': '抗老不是越贵越有效，基础护肤也很重要！', 'content_form': '清单+教程', 'risk_note': '不

In [10]:
angle_strategist_prompt = """
【角色】
你是“小红书护肤内容策划”。

【目标】
为同一个护肤选题设计 3 个不同的传播切入角度，
让内容更像真实小红书经验分享。

【硬性约束（必须遵守）】
- 每个角度都要明确：读者“看完能得到什么实际收获”。
- 使用第一人称或经验型视角，但不得虚构医学背景。
- 禁止绝对化、治疗承诺、医疗术语堆砌。

【输出要求（必须严格JSON数组）】
请为每个选题输出 3 个角度，每个角度包含：
- angle_name
- opening_hook
- value_promise
- suggested_structure

格式如下:
[
  {"topic_index_1": [
      {"angle_name": string, "opening_hook": string, "value_promise": string, "suggested_structure": string},
      {"angle_name": string, "opening_hook": string, "value_promise": string, "suggested_structure": string},
      {"angle_name": string, "opening_hook": string, "value_promise": string, "suggested_structure": string},
  ]},
  {"topic_index_2": [
      {"angle_name": string, "opening_hook": string, "value_promise": string, "suggested_structure": string},
      {"angle_name": string, "opening_hook": string, "value_promise": string, "suggested_structure": string},
      {"angle_name": string, "opening_hook": string, "value_promise": string, "suggested_structure": string},
  ]},
  {"topic_index_3": [
      {"angle_name": string, "opening_hook": string, "value_promise": string, "suggested_structure": string},
      {"angle_name": string, "opening_hook": string, "value_promise": string, "suggested_structure": string},
      {"angle_name": string, "opening_hook": string, "value_promise": string, "suggested_structure": string},
  ]}
  ...
]
"""

In [11]:
messages2 = [
    SystemMessage(content=angle_strategist_prompt), 
    HumanMessage(content=f"读取候选选题列表: {trend_options}，根据system规则生成传播角度")
]
response2 = model.invoke(messages2)

In [12]:
angle_options = json.loads(response2.content)

In [13]:
print(angle_options)

[{'topic_index_1': [{'angle_name': '亲测：洗脸次数多反而出油？我的油皮清洁避坑经验', 'opening_hook': '以前我以为多洗几次脸就能控油，结果皮肤越来越油还爆痘…', 'value_promise': '看完你能避开常见洗脸误区，找到适合自己的清洁频率和方法。', 'suggested_structure': '1. 讲述自己曾经频繁洗脸的经历和后果；2. 总结常见清洁误区（如次数、力度、产品选择）；3. 分享调整后更适合油皮/混合皮的洗脸方法；4. 总结适合不同肤质的清洁建议。'}, {'angle_name': '对比实测：一天洗两次VS三次，皮肤状态大不同', 'opening_hook': '我专门试了一个星期，一天洗两次和三次，皮肤变化超出预期！', 'value_promise': '你能直观看到不同洗脸频率对皮肤的影响，学会科学安排清洁。', 'suggested_structure': '1. 记录对比实验过程（洗脸次数、皮肤感受）；2. 展示皮肤状态变化（如干燥、出油、爆痘）；3. 分析原因，结合常见误区；4. 给出适合学生党/油皮的清洁建议。'}, {'angle_name': '洗脸误区盘点：这些小细节才是皮肤变差的元凶', 'opening_hook': '你是不是也有这些洗脸习惯？其实很多都是误区！', 'value_promise': '帮你识别日常清洁中容易忽略的细节，避免踩坑让皮肤更稳定。', 'suggested_structure': '1. 罗列常见洗脸误区（如水温、用力搓、洗面奶用量）；2. 结合自身经历举例说明；3. 分享正确做法和调整后的变化；4. 总结适合不同肤质的清洁tips。'}]}, {'topic_index_2': [{'angle_name': '油皮也要保湿？我的保湿误区和正确打开方式', 'opening_hook': '以前我觉得油皮不用保湿，结果越控油越出油…', 'value_promise': '你能了解不同肤质的保湿需求，找到适合自己的保湿方法。', 'suggested_structure': '1. 分享自己对保湿的误解和踩过的坑；2. 讲解油皮、干皮、敏感皮的保湿重点；3. 推荐简单易行的保湿步骤；4. 总结保湿小技巧。'}, {'angl

In [14]:
virality_score_prompt = """
【角色】
你是“小红书护肤内容增长分析师”，只负责评估传播潜力，不写正文。

【目标】
评估“选题 + 切入角度”的传播潜力，打分并排序，输出 Top3。

【评分维度】
- click_potential
- save_value
- comment_potential
- execution_barrier
- compliance_risk

【输出要求（必须严格JSON数组）】
[{
  "total_score": number,
  "breakdown": {
    "click_potential": number,
    "save_value": number,
    "comment_potential": number,
    "execution_barrier": number,
    "compliance_risk": number
  },
  "strengths": string[],
  "weaknesses": string[],
  "optimization_suggestions": string[],
  "topic_index": string,
  "topic": string,
  "angle_name": string
},
{
  "total_score": number,
  "breakdown": {
    "click_potential": number,
    "save_value": number,
    "comment_potential": number,
    "execution_barrier": number,
    "compliance_risk": number
  },
  "strengths": string[],
  "weaknesses": string[],
  "optimization_suggestions": string[],
  "topic_index": string,
  "topic": string,
  "angle_name": string
},
{
  "total_score": number,
  "breakdown": {
    "click_potential": number,
    "save_value": number,
    "comment_potential": number,
    "execution_barrier": number,
    "compliance_risk": number
  },
  "strengths": string[],
  "weaknesses": string[],
  "optimization_suggestions": string[],
  "topic_index": string,
  "topic": string,
  "angle_name": string
}]
"""

In [15]:
messages3 = [
    SystemMessage(content=virality_score_prompt), 
    HumanMessage(content=f"读取候选选题列表: {trend_options}，以及对应的每个选题的传播切入角度列表{angle_options}，根据“选题 + 切入角度”的传播潜力，做评估，")
]
response3 = model.invoke(messages3)

In [16]:
scores_options = json.loads(response3.content)

In [17]:
scores_options

[{'total_score': 44,
  'breakdown': {'click_potential': 9,
   'save_value': 9,
   'comment_potential': 9,
   'execution_barrier': 8,
   'compliance_risk': 9},
  'strengths': ['对比实验形式极具吸引力，容易激发好奇心和点击欲望',
   '内容结构清晰，数据和体验结合，易于被用户保存和复用',
   '话题贴近学生党、油皮群体的真实困惑，评论互动空间大'],
  'weaknesses': ['需要一定周期的亲测和记录，执行门槛略高', '对皮肤状态变化的展示需真实，避免夸大效果'],
  'optimization_suggestions': ['可增加皮肤状态对比的图片或数据，提升说服力', '强调个体差异，避免一刀切结论'],
  'topic_index': 'topic_index_1',
  'topic': '洗脸次数越多越干净？日常清洁的3个误区',
  'angle_name': '对比实测：一天洗两次VS三次，皮肤状态大不同'},
 {'total_score': 43,
  'breakdown': {'click_potential': 9,
   'save_value': 9,
   'comment_potential': 8,
   'execution_barrier': 8,
   'compliance_risk': 9},
  'strengths': ['‘厚涂面霜≠保湿’观点新颖，易引发用户兴趣和讨论',
   '针对不同肤质给出清单，实用性强，易于保存',
   '避坑属性强，适合换季、敏感等高需求场景'],
  'weaknesses': ['需注意内容不涉及具体品牌或成分，避免合规风险', '部分用户可能对‘厚涂’误区已有认知，需突出新信息'],
  'optimization_suggestions': ['可结合真实案例或用户反馈，增强说服力', '适当补充保湿原理，提升专业性'],
  'topic_index': 'topic_index_2',
  'topic': '保湿≠厚涂面霜！不同肤质的保湿方案清单',
  'angle_nam

In [18]:
outline_prompt = """
【角色】
你是“小红书护肤笔记结构工程师”。

【目标】
把护肤主题转化为“高完读率、高保存率”的图文大纲。

【硬性约束（必须遵守）】
- 必须包含：3秒开头钩子 → 核心结论 → 分点展开（步骤/清单/对比）→ 总结卡片 → 互动提问。
- 至少提供 1 个可复制模板（如早晚护肤流程、成分避坑表、搭配公式）。
- 内容为日常护理建议，不是医学结论；涉及痘痘/敏感只给温和护理建议，并建议必要时咨询皮肤科。

【输出要求】
请输出 Markdown 大纲 outline，使用：
- ## 作为一级结构
- 列表呈现步骤或清单

格式如下(严格JSON数组):
请输出一个 JSON 数组，每个元素对应一篇 outline，结构如下：
[
  {
    "outline_id": "string",
    "outline_md": "string"
  }
]
"""

In [20]:
scores_options[0]["topic"],scores_options[0]["angle_name"]

('洗脸次数越多越干净？日常清洁的3个误区', '对比实测：一天洗两次VS三次，皮肤状态大不同')

In [21]:
scores_options[1]["topic"],scores_options[1]["angle_name"]

('保湿≠厚涂面霜！不同肤质的保湿方案清单', '厚涂面霜≠保湿？三种肤质的保湿方案清单')

In [19]:
messages4 = [
    SystemMessage(content=outline_prompt), 
    HumanMessage(content=f"读取传播潜力top3列表: {scores_options}，在选题列表{trend_options}以及传播切入角度列表{angle_options}中找到对应的主题详情和切入角度详情，分别为top3选题和对应角度{scores_options}生成图文大纲")
]
response4 = model.invoke(messages4)

In [22]:
outline_list = json.loads(response4.content)

In [23]:
outline_list

[{'outline_id': 'topic_index_1_angle_2',
  'outline_md': '## 3秒开头钩子\n- 我专门试了一个星期，一天洗两次和三次，皮肤变化超出预期！\n\n## 核心结论\n- 洗脸次数并非越多越好，过度清洁反而可能让皮肤更油、更敏感。找到适合自己的清洁频率，皮肤状态才会更稳定。\n\n## 分点展开\n### 1. 对比实测过程\n- 实验周期：7天\n- 分组：前3天每天洗脸2次（早晚各一次），后4天每天洗脸3次（早中晚）\n- 记录内容：皮肤出油、干燥、爆痘、紧绷感等变化\n\n### 2. 皮肤状态对比\n- 洗两次：\n  - 皮肤出油量适中，T区略油但不爆痘\n  - 整体水油平衡，脸颊无紧绷感\n- 洗三次：\n  - 第2天起T区明显更油，偶有小痘点\n  - 脸颊有轻微干燥、紧绷，偶尔泛红\n\n### 3. 原因分析&常见误区\n- 误区1：洗脸越多越干净\n  - 实际：频繁清洁会破坏皮脂膜，皮肤反而更容易出油或敏感\n- 误区2：用力搓洗、追求“洗到吱嘎”\n  - 正确做法：温和打圈，避免过度摩擦\n- 误区3：忽略水温和洗面奶用量\n  - 建议：用温水，洗面奶用量适中即可\n\n### 4. 针对油皮/学生党的清洁建议\n- 日常建议：\n  - 早晚各一次，运动后可适当加一次（如有大量出汗）\n  - 选择温和型洗面奶，避免强碱性或去油力过强产品\n  - 洗后及时补水保湿，维持皮肤屏障\n\n## 总结卡片\n- 洗脸不是越多越好，适合自己的频率才最重要。过度清洁=皮肤屏障受损，油皮也要温和对待！\n\n## 互动提问\n- 你一天洗几次脸？有没有遇到过洗太多反而更油的情况？欢迎留言分享你的清洁习惯！\n\n---\n### 可复制模板：日常洗脸流程\n1. 用温水打湿面部\n2. 取适量温和洗面奶，揉出泡沫\n3. 轻柔打圈清洁全脸（T区可多打圈，U区轻柔带过）\n4. 用清水彻底冲洗干净\n5. 轻拍毛巾吸干水分，及时补水保湿\n\n> 温馨提示：如有痘痘/敏感，仅建议温和清洁，必要时咨询皮肤科医生。'},
 {'outline_id': 'topic_index_2_angle_2',
  'outline_md': '## 3秒开头钩子\n- 保湿不是面霜越厚越好！不同肤质其

In [26]:
draft_prompt = """
【角色】
你是“小红书护肤图文写手（Draft Writer）”。

【定位说明（非常重要）】
你正在参与一个多候选内容竞争流程：
- 每一篇初稿（draft）都将与其他初稿一起被比较、评分和筛选
- 只有一篇会进入后续的深度打磨与发布流程
因此，你写的每一篇 draft 都必须是“完整、可发布、可比较”的版本。

【目标】
你将收到一个名为 outline_list 的图文大纲列表。
outline_list 中的每一条 outline 都代表【一篇独立的护肤笔记候选】。
请你为 outline_list 中的【每一条 outline】分别撰写一篇完整的小红书护肤图文初稿。

【输入说明】
- outline_list 是一个列表
- 每一条 outline 都已包含：开头钩子、核心结论、展开要点、总结卡片、互动提问
- 不同 outline 之间内容不同，请分别独立完成，不要混写

【硬性约束（必须遵守）】
- 中文输出，短句、短段落，强扫读，适合手机阅读
- 不得出现外链、联系方式、私信引导或任何引流话术
- 禁止治疗承诺、绝对化功效或医学结论
- 成分与机理仅限大众理解层面，不编造研究数据
- 内容必须真实、克制，像真实用户经验分享
- 不要在正文中提及“大纲”“outline”“列表结构”等提示性词语
- 不要简单复述 outline 的标题结构，而是写成自然流畅的正文

【正文内容要求（每一篇都必须包含）】
- 共情型开头（贴近日常护肤困扰）
- 清单 / 步骤 / 对比（给出可操作细节）
- 明确的个人体验或感受（不夸大）
- 结尾互动问题（引导评论分享经验）
- ✅ 一屏总结（适合截图保存）

【输出要求（必须严格 JSON；不要输出多余文字）】
请输出一个 JSON 数组，每个元素对应一篇 draft，结构如下：
[
  {
    "draft_id": "string",
    "draft_md": "string"
  }
]

【额外要求】
- draft_id 必须与对应 outline 的唯一标识一致（如 outline_id / outline_index）
- draft_md 必须是完整 Markdown 正文
- 输出数组的元素数量必须与 outline_list 的数量一致
"""

In [27]:
messages5 = [
    SystemMessage(content=draft_prompt), 
    HumanMessage(content=f"这是图文大纲 outline_list（JSON）：{outline_list}, 请按 system 规则生成初稿。 ")
]
response5 = model.invoke(messages5)

In [28]:
draft_list = json.loads(response5.content)

In [29]:
print(draft_list[0])

{'draft_id': 'topic_index_1_angle_2', 'draft_md': '你是不是也觉得，脸洗得越勤越干净？我专门试了一个星期，一天洗两次和三次，结果真的有点出乎意料！\n\n先说结论：洗脸次数多不一定更好，反而可能让皮肤更油、更敏感。找到适合自己的清洁频率，皮肤状态才会更稳定。\n\n我的实测过程是这样的：前3天每天洗两次（早晚各一次），后4天每天洗三次（早中晚）。每天都认真记录皮肤的出油、干燥、爆痘和紧绷感。\n\n洗两次的时候，T区出油量适中，偶尔有点油但没爆痘，脸颊也没有紧绷感，整体水油挺平衡。\n\n但一到每天三次，第二天起T区就明显更油了，还冒了几个小痘点。脸颊反而有点干、紧绷，偶尔还泛红。\n\n其实很多人都有误区：\n- 以为洗脸越多越干净，其实频繁清洁会破坏皮脂膜，皮肤反而更容易出油或敏感。\n- 用力搓洗、追求“洗到吱嘎”，但这样只会让皮肤更受伤。正确做法是温和打圈，避免过度摩擦。\n- 忽略水温和洗面奶用量，建议用温水，洗面奶适量就好。\n\n如果你是油皮或者学生党，建议：\n- 日常早晚各一次，运动后大量出汗可以适当加一次\n- 选温和型洗面奶，别用去油力太强的\n- 洗完脸要及时补水保湿，维持皮肤屏障\n\n**一屏总结：**\n- 洗脸不是越多越好，适合自己的频率才最重要。过度清洁=皮肤屏障受损，油皮也要温和对待！\n\n你一天洗几次脸？有没有遇到过洗太多反而更油的情况？欢迎留言分享你的清洁习惯！\n\n---\n\n### 日常洗脸流程（可截图保存）\n1. 用温水打湿面部\n2. 取适量温和洗面奶，揉出泡沫\n3. 轻柔打圈清洁全脸（T区可多打圈，U区轻柔带过）\n4. 用清水彻底冲洗干净\n5. 轻拍毛巾吸干水分，及时补水保湿\n\n> 温馨提示：如有痘痘/敏感，仅建议温和清洁，必要时咨询皮肤科医生。'}


In [31]:
title_prompt = """
【角色】
你是“小红书护肤标题与钩子实验室（Hook & Title Lab）”。

【定位说明（非常重要）】
你不是在“选标题”，而是在为后续的 Draft+Title Ranker 提供
【可比较、可筛选、可评分】的标题候选池。

【目标】
你将收到一个名为 draft_list 的初稿列表。
draft_list 中包含多篇不同的护肤图文初稿（每篇代表一个独立候选）。
请为【每一篇初稿】分别生成多组易于传播的小红书标题与封面钩子文案，
以最大化点击欲望、保存价值和完读率。

【输入说明】
- 每个 draft 至少包含：draft_id、draft_md
- 不同 draft 之间内容不同，标题必须严格对应各自 draft 内容
- 不允许将不同 draft 的信息混入同一个标题

【硬性约束（必须遵守）】
- 标题必须真实、克制，不夸大、不医疗化、不制造恐慌
- 禁止使用或暗示以下词语：
  “立刻 / 根治 / 永久 / 医学证明 / 百分百 / 治好 / 治疗”
- 不得出现品牌名、产品名、价格、广告或明显商业导向
- 不得出现外链、联系方式、私信引导
- 不得承诺 draft_md 中未明确支持的效果或结果
- 同一篇 draft 内，标题之间不得高度重复（避免仅做同义改写）

【标题句式覆盖要求（每一篇 draft 都必须覆盖）】
请至少覆盖以下标题类型，并为每个标题标注 type：
1) avoid     —— 避坑型（别再… / 很多人都在…）
2) compare   —— 对比型（以前…现在… / 做错 VS 做对）
3) list      —— 清单型（3个方法 / 5个关键点）
4) scenario  —— 场景型（换季 / 油皮夏天 / 敏感期）
5) result    —— 结果导向型（更稳定 / 更清爽 / 更耐受）
※ result 类型只允许使用“更……”，不得使用“见效 / 改善明显 / 解决问题”等绝对化表述。

【输出要求（必须严格 JSON；不要输出多余文字）】
{
  "candidates": [
    {
      "draft_id": "string",
      "titles": [
        {
          "title": "string",
          "type": "avoid | compare | list | scenario | result",
          "core_hook": "string"
        }
      ],
      "cover_copies": ["string"]
    }
  ]
}

【数量要求】
- 对每一篇 draft：
  - titles：15–20 条
  - cover_copies：5 条（单条 ≤12 字）
- results 的条目数量必须与 draft_list 的篇数一致

【质量要求（为后续 Ranker 服务）】
- 标题之间要有明显差异，便于比较优劣
- core_hook 必须明确该标题的主要吸引点（避坑 / 对比 / 场景 / 结果）
- 不要试图“帮我选最好的一条”，筛选由后续 Ranker 完成
"""

In [32]:
messages6 = [
    SystemMessage(content=title_prompt), 
    HumanMessage(content=f"这是初稿列表 draft_list（JSON）：{draft_list}, 请按 system 规则生成小红书标题与封面钩子文案。 ")
]
response6 = model.invoke(messages6)

In [33]:
candidates = json.loads(response6.content)

In [34]:
candidates

{'candidates': [{'draft_id': 'topic_index_1_angle_2',
   'titles': [{'title': '别再频繁洗脸了！我实测一天洗三次的真实后果',
     'type': 'avoid',
     'core_hook': '频繁洗脸反而让皮肤更油更敏感'},
    {'title': '洗脸次数多VS少，皮肤状态差别有多大？我的一周实测',
     'type': 'compare',
     'core_hook': '对比不同洗脸频率对皮肤的影响'},
    {'title': '洗脸避坑清单：3个常见误区你中招了吗？',
     'type': 'list',
     'core_hook': '总结洗脸常见误区，帮助避坑'},
    {'title': '油皮学生党洗脸频率怎么选？我的实测建议',
     'type': 'scenario',
     'core_hook': '针对油皮和学生党场景的洗脸建议'},
    {'title': '更稳定的皮肤状态，原来靠这一步：适合自己的洗脸频率',
     'type': 'result',
     'core_hook': '找到合适洗脸频率让皮肤更稳定'},
    {'title': '很多人都以为洗脸越多越干净？其实可能更油！',
     'type': 'avoid',
     'core_hook': '揭示洗脸次数多的反效果'},
    {'title': '以前一天三次洗脸，现在只洗两次，皮肤变化超明显',
     'type': 'compare',
     'core_hook': '对比洗脸次数前后的皮肤变化'},
    {'title': '5步日常洗脸流程，皮肤不紧绷不爆痘', 'type': 'list', 'core_hook': '详细分解科学洗脸流程'},
    {'title': '换季时洗脸要注意什么？我的亲测经验分享',
     'type': 'scenario',
     'core_hook': '换季场景下的洗脸注意事项'},
    {'title': '更平衡的水油状态，洗脸次数其实很关键',
     'type': 'result',
     '

In [35]:
draft_title_ranker_prompt = """
【角色】
你是“小红书护肤增长总监 + 总编联合评审官（Draft+Title Ranker）”。

【目标】
你将收到两份输入：
1) draft_list：包含多篇候选笔记初稿（每篇有 draft_id 与 draft_md）
2) candidates：包含对应 draft_id 的标题候选池（titles）与封面大字（cover_copies）

你的任务是：在所有候选中筛选出【唯一】最值得发布的文章，并为它选择【唯一】最优标题。
选择标准：传播潜力最大（点击+收藏+完读）且完全合规、真实克制、不夸大。

【输入说明】
- draft_list 与 candidates 通过 draft_id 一一对应
- titles 中可能包含 type（avoid/compare/list/scenario/result）与 core_hook（吸引点）
- 你必须同时评估“内容质量（draft_md）”与“标题潜力（titles/cover_copies）”
- 不允许把不同 draft 的信息混在一起判断

【硬性约束（必须遵守）】
- 必须真实、克制，不医疗化、不制造恐慌
- 禁止选择包含或暗示以下内容的标题或表达：
  “立刻/根治/永久/医学证明/百分百/治好/治疗/处方/消炎/药到病除”等
- 不得引导私信加联系方式，不得外链，不得出现品牌广告导向
- 标题必须与 draft_md 内容一致；若标题承诺点在正文中无支撑，视为严重扣分
- 若所有候选都有风险，你必须选风险最低者，并在输出中给出“必须修改清单”

【评审维度与权重（每篇 0-1 分，保留两位小数）】
你必须对每个 draft_id 输出以下分数，并给出简短依据：
1) click_score（点击欲望：标题池强度+封面大字潜力）—— 0.25
2) save_score（收藏价值：是否有清单/步骤/模板/对比/一屏总结）—— 0.25
3) readability_score（扫读与完读：结构清晰、短句短段、信息密度高）—— 0.20
4) authenticity_score（真实感与克制：像真实经验分享，不夸大，不“科普腔”）—— 0.15
5) compliance_score（合规安全：不医疗化、不引流、不夸大、不敏感）—— 0.15

总分 total_score = 加权求和。

【标题选择规则（非常重要）】
对最终胜出的 best_draft_id：
- 从该 draft 的 titles 中选出 best_title（最有传播潜力且合规）
- 如 best_title 存在“轻微风险/表达过度/承诺过强”，必须提供 safer_title（更克制更安全的版本）
- 同时给出 best_cover_copy（从 cover_copies 中选 1 条最适合封面大字）

【输出要求（必须严格 JSON；不要输出多余文字）】
{
  "ranking": [
    {
      "draft_id": "string",
      "total_score": number,
      "scores": {
        "click_score": number,
        "save_score": number,
        "readability_score": number,
        "authenticity_score": number,
        "compliance_score": number
      },
      "reason": "string",
      "best_title_for_this_draft": "string",
      "best_cover_copy_for_this_draft": "string",
      "title_risk_notes": ["string"],
      "must_fix_if_selected": ["string"]
    }
  ],
  "winner": {
    "best_draft_id": "string",
    "best_title": "string",
    "safer_title": "string",
    "best_cover_copy": "string",
    "why_win": ["string"],
    "must_fix_before_R1": ["string"],
    "optional_improvements": ["string"]
  }
}

【额外要求】
- ranking 必须包含 draft_list 中每一篇（数量一致）
- total_score 与各项 scores 必须保留两位小数
- reason/why_win/must_fix 必须具体可执行（避免空泛表述）
- 如果某篇 draft_md 缺少“✅一屏总结”或“互动问题”，save_score/readability_score 必须明显扣分并写入 must_fix_if_selected
"""

In [36]:
messages7 = [
    SystemMessage(content=draft_title_ranker_prompt), 
    HumanMessage(content=f"这是初稿列表 draft_list（JSON）：{draft_list}, 这是candidates列表 candidates（JSON）：{candidates}, 请按 system 规则进行评审并选择最佳。 ")
]
response7 = model.invoke(messages7)

In [40]:
winner = json.loads(response7.content)

In [41]:
winner["winner"]

{'best_draft_id': 'topic_index_2_angle_2',
 'best_title': '3种肤质保湿方案清单，照着做不闷痘',
 'safer_title': '3种肤质保湿方案清单，照着做更安心',
 'best_cover_copy': '分肤质保湿方案',
 'why_win': ['内容针对性强，分肤质方案易于收藏和转发，表格清单提升一屏总结感',
  '标题池中“清单”“分肤质”类标题强，封面大字直观，易吸引点击',
  '表达克制，合规性好，无夸大或医疗化，真实感强',
  '扫读性和互动性均佳，适合广泛传播'],
 'must_fix_before_R1': ['将标题中的“不闷痘”调整为更克制表达，如“更安心”', '结尾互动问题可更具体引导用户分享自身保湿习惯'],
 'optional_improvements': ['表格可用更直观配色或符号提升扫读体验', '增加一条“常见保湿误区”小结，增强收藏价值']}

In [75]:
r1_reflector_prompt = """
【角色】
你是“小红书护肤内容总编（R1 Reflector）”。

【定位说明（非常重要）】
你处理的是【已经被选中的唯一胜出稿件】。
你的任务不是重新选题，而是通过专业编辑反思与改稿，
将当前稿件提升到“编辑部可发布水准”。

【目标】
你将收到：
- best_draft_id
- 当前 draft_md（正文 Markdown）
- best_title（已选标题）
- must_fix_before_R1（必须修复的问题清单）
- optional_improvements（可选提升建议）

请在不改变选题方向、不夸大的前提下，
通过结构优化、表达优化、信息补强与真实感增强，
显著提升内容的【清晰度 / 保存价值 / 完读率 / 可信度】。

【硬性约束（必须遵守）】
- 不改变文章的核心观点与结论方向
- 不引入任何新的未经正文支持的结论
- 不医疗化、不制造恐慌、不使用绝对化表述
- 不出现外链、联系方式、私信引导
- 不加入品牌名、广告或商业导向
- 保持小红书用户真实经验分享的语气

【编辑优先级（必须按顺序执行）】
1) 必须优先修复 must_fix_before_R1 中列出的所有问题
2) 在不破坏整体节奏的前提下，参考 optional_improvements 提升质量
3) 确保正文包含并强化以下结构要素：
   - 共情型开头（更贴近日常护肤真实困扰）
   - 清晰可复制的清单 / 步骤 / 对比
   - 明确的一屏总结（适合截图保存）
   - 自然、不生硬的结尾互动问题
4)如果best_title有进行修改，将新title输出到变量revised_title，如果没有进行任何修改，变量revised_title使用best_title原值

【评分维度（用于是否进入下一轮）】
请对当前版本给出以下评分（0–1）：
- clarity_score（结构与表达是否清晰）
- save_value_score（是否值得收藏）
- readability_score（扫读与完读友好度）
- authenticity_score（真实感与克制程度）

【输出要求（必须严格 JSON；不要输出多余文字）】
{
  "draft_id": "string",
  "revised_title": "string",
  "revised_md": "string",
  "scores": {
    "clarity_score": number,
    "save_value_score": number,
    "readability_score": number,
    "authenticity_score": number
  },
  "fixed_issues": ["string"],
  "remaining_risks": ["string"],
  "editor_notes": ["string"],
  "should_run_R1_again": boolean
}

【循环建议】
- 如果 should_run_R1_again 为 true，
  表示仍存在明显结构或表达问题，可再进行一轮 R1 反思
- 如果为 false，
  表示内容质量已达编辑部可交付标准，可进入后续节点
"""

In [76]:
winner["winner"]["best_draft_id"]

'topic_index_2_angle_2'

In [55]:
draft_list[1]

{'draft_id': 'topic_index_2_angle_2',
 'draft_md': '是不是觉得保湿就是面霜越厚越好？其实不同肤质，保湿方案真的差很大！\n\n我的经验是：保湿≠厚涂面霜。根据肤质选择合适的保湿方式，皮肤才能真正水润不闷痘。\n\n很多人以为面霜涂得越厚越保湿，但其实厚涂反而容易堵塞毛孔、闷痘，皮肤反而不舒服。\n\n不同肤质的保湿需求也不一样：\n- 干皮需要锁水+滋润，注重分层保湿\n- 油皮更适合轻薄补水，避免油腻厚重\n- 敏感皮要温和无刺激，减少成分负担\n\n我的保湿流程清单：\n\n**干皮**\n1. 温和清洁\n2. 补水型化妆水\n3. 保湿精华/乳液\n4. 滋润型面霜（薄涂多次叠加，不要一次厚涂）\n\n**油皮**\n1. 温和清洁\n2. 清爽型化妆水\n3. 轻薄型乳液/凝露\n4. 局部干燥可点涂面霜\n\n**敏感皮**\n1. 温和清洁（避免香精、酒精）\n2. 舒缓型化妆水\n3. 简单成分的保湿乳/霜\n4. 避免频繁更换产品\n\n我的感受是，保湿要分层、少量多次，避免一次性厚涂。换季或皮肤敏感时，优先选温和、成分简单的产品。油皮也不能忽略保湿，否则皮肤反而更容易出油。\n\n**一屏总结：**\n- 保湿不是越厚越好，分肤质、分步骤才是关键。科学保湿，让皮肤水润不闷痘！\n\n你平时保湿会厚涂面霜吗？你的肤质适合哪种保湿方案？欢迎留言讨论！\n\n---\n\n### 三种肤质保湿清单（可截图保存）\n| 肤质   | 清洁      | 补水      | 保湿      | 特别注意         |\n|--------|-----------|-----------|-----------|------------------|\n| 干皮   | 温和型    | 补水型水  | 滋润面霜  | 分层叠加，薄涂  |\n| 油皮   | 温和型    | 清爽型水  | 轻薄乳液  | 局部干燥点涂    |\n| 敏感皮 | 温和无刺激| 舒缓型水  | 简单乳/霜 | 避免频繁更换    |\n\n> 温馨提示：如遇持续干燥、爆皮或敏感，建议咨询皮肤科医生。'}

In [48]:
best_draft_id = winner["winner"]["best_draft_id"]

In [56]:
best_draft_md = [d["draft_md"] for d in draft_list if d["draft_id"] == best_draft_id ][0]

In [58]:
best_title = winner["winner"]["best_title"]

In [59]:
must_fix_before_R1 = winner["winner"]["must_fix_before_R1"]

In [64]:
optional_improvements = winner.get("winner").get("optional_improvements") or [""]

In [77]:
messages8 = [
    SystemMessage(content=r1_reflector_prompt), 
    HumanMessage(content=f"这是best_draft_id：{best_draft_id}, 这是best_draft_md：{best_draft_md}, 这是best_title：{best_title}，这是must_fix_before_R1：{must_fix_before_R1}， 这是optional_improvements： {optional_improvements}，请按 system 规则进行反思与改稿。 ")
]
response8 = model.invoke(messages8)

In [78]:
best_title

'3种肤质保湿方案清单，照着做不闷痘'

In [84]:
reflected_draft = json.loads(response8.content)

In [85]:
reflected_draft

{'draft_id': 'topic_index_2_angle_2',
 'revised_title': '3种肤质保湿方案清单，照着做更安心',
 'revised_md': '是不是觉得保湿就是面霜越厚越好？其实不同肤质，保湿方案真的差很大！\n\n我自己也踩过不少坑，后来才发现：保湿≠厚涂面霜。只有根据肤质选择合适的保湿方式，皮肤才能真正水润、不闷、不负担。\n\n很多人以为面霜涂得越厚越保湿，但其实厚涂反而容易堵塞毛孔、闷痘，皮肤反而不舒服。\n\n**不同肤质的保湿需求也不一样：**\n- 🟤 干皮：需要锁水+滋润，注重分层保湿\n- 🟢 油皮：更适合轻薄补水，避免油腻厚重\n- 🟡 敏感皮：要温和无刺激，减少成分负担\n\n---\n\n### 我的保湿流程清单\n\n**干皮**\n1. 温和清洁\n2. 补水型化妆水\n3. 保湿精华/乳液\n4. 滋润型面霜（薄涂多次叠加，不要一次厚涂）\n\n**油皮**\n1. 温和清洁\n2. 清爽型化妆水\n3. 轻薄型乳液/凝露\n4. 局部干燥可点涂面霜\n\n**敏感皮**\n1. 温和清洁（避免香精、酒精）\n2. 舒缓型化妆水\n3. 简单成分的保湿乳/霜\n4. 避免频繁更换产品\n\n---\n\n**常见保湿误区小结：**\n- 只追求厚涂面霜，忽略分层和肤质差异\n- 油皮觉得不用保湿，结果皮肤更容易出油\n- 敏感皮频繁换产品，反而加重不适\n\n我的感受是，保湿要分层、少量多次，避免一次性厚涂。换季或皮肤敏感时，优先选温和、成分简单的产品。油皮也不能忽略保湿，否则皮肤反而更容易出油。\n\n---\n\n**一屏总结（可截图保存）：**\n- 保湿不是越厚越好，分肤质、分步骤才是关键。科学保湿，让皮肤更安心！\n\n---\n\n### 三种肤质保湿清单\n| 肤质   | 清洁      | 补水      | 保湿      | 特别注意         |\n|--------|:---------:|:---------:|:---------:|:----------------:|\n| 🟤 干皮   | 温和型    | 补水型水  | 滋润面霜  | 分层叠加，薄涂  |\n| 🟢 油皮   | 温和型    | 清爽型水  | 轻薄乳液  | 局部干燥点涂   

In [86]:
r2_compliance_prompt = """
【角色】
你是“小红书护肤内容合规官（R2 Compliance Officer）”。

【定位说明（非常重要）】
你是内容发布前的最后一道合规审查节点。
你的职责不是提升传播效果，也不是润色文案，
而是从平台规则、内容安全与风险控制角度，
判断当前内容是否【可以安全发布】。

【目标】
你将收到：
- draft_id
- 当前正文 revised_md（Markdown）
- 当前拟用标题 best_title
- 备选安全标题 safer_title
- R1 Reflector 输出的 remaining_risks（如有）

请你在不考虑“传播效果”的前提下，
仅从合规与安全角度进行严格审查。

【审查范围（必须逐项检查）】
1) 医疗与功效风险
   - 是否构成疾病诊断、治疗、医疗建议暗示
   - 是否出现“治好/治疗/消炎/修复受损组织”等医疗化表述
2) 绝对化与夸大表述
   - 是否出现“立刻/根治/永久/百分百/必然/一定会”等
3) 平台违规与引流风险
   - 外链、联系方式、私信引导、二维码暗示
4) 误导性内容
   - 标题是否承诺正文中未充分支持的效果
   - 是否容易被普通用户误解为“专业医疗建议”
5) 侵权与敏感风险
   - 是否暗含品牌对比、功效暗示、群体歧视、恐吓式表达

【决策原则（必须遵守）】
- 风险优先级高于传播效果
- 如 best_title 存在风险，应优先考虑 safer_title
- 若正文存在结构性风险，必须要求修改，不得“带病放行”

【输出要求（必须严格 JSON；不要输出多余文字）】
{
  "pass": boolean,
  "final_title": "string",
  "issues": [
    {
      "type": "medical | exaggeration | misleading | platform | infringement | other",
      "description": "string",
      "severity": "low | medium | high"
    }
  ],
  "required_fixes": ["string"],
  "suggested_fixes": ["string"],
  "notes_for_editor": ["string"]
}

【决策说明】
- pass = true：
  - 表示当前内容在修复 required_fixes 后可以发布
  - final_title 必须给出（通常为 best_title 或 safer_title）
- pass = false：
  - 表示存在高风险，不应进入发布流程
  - required_fixes 必须明确指出需要回到 R1 修改的内容
"""

In [106]:
draft_id = reflected_draft["draft_id"]

In [90]:
revised_md = reflected_draft["revised_md"]

In [89]:
best_title = reflected_draft["revised_title"]

In [88]:
safer_title = winner["winner"]["safer_title"]

In [91]:
best_title, safer_title

('3种肤质保湿方案清单，照着做更安心', '3种肤质保湿方案清单，照着做更安心')

In [105]:
remaining_risks = reflected_draft.get("remaining_risks") or [""]

In [108]:
messages9 = [
    SystemMessage(content=r2_compliance_prompt), 
    HumanMessage(content=f"这是draft_id：{draft_id}, 这是当前正文revised_md：{revised_md}, 这是当前拟用标题 best_title：{best_title}，这是备选安全标题 safer_title：{safer_title}， 这是R1 Reflector 输出的 remaining_risks： {remaining_risks}，请按 system 规则进行合规审查。 ")
]
response9 = model.invoke(messages9)

In [111]:
r2_complianced = json.loads(response9.content)

In [122]:
r2_complianced

{'pass': True,
 'final_title': '3种肤质保湿方案清单，照着做更安心',
 'issues': [],
 'required_fixes': [],
 'suggested_fixes': ['正文已在结尾处提示如遇持续干燥、爆皮或敏感建议咨询医生，建议持续保留此类提示。',
  '如后续补充具体产品或成分，需避免医疗化、绝对化表述。'],
 'notes_for_editor': ['内容未涉及医疗诊断、治疗或功效承诺，表述以个人经验和常见误区为主，风险可控。',
  '未出现绝对化、夸大、引流、侵权、敏感等违规风险。',
  '如后续添加产品推荐或更具体建议，需再次进行合规审查。']}

In [128]:
final_title = r2_complianced["final_title"]

In [127]:
visual_prompt = """
【角色】
你是“小红书护肤视觉导演（Visual Director）”。

【定位说明】
你负责根据标题与正文内容制定配图策略，并输出“配图脚本”（image script）。
你不负责真正下载图片；执行由后续 Image Sourcing 节点完成。

【目标】
你将收到：
- final_title（最终标题）
- revised_md（已编辑后的正文 Markdown）
请你为该笔记设计一组配图，用于提升点击、完读与收藏。
风格要求：实拍 / 简约（干净、自然光、低饱和、生活场景）。
避免广告感，避免品牌露出。

【硬性约束（必须遵守）】
- 图片必须与正文内容一致，不得引入新的夸大结论
- 不出现品牌 Logo、产品商标、价格标签
- 尽量避免可识别的人脸（除非明确需要“生活方式氛围”，且应模糊/背影/手部）
- 不出现医疗诊疗场景、手术、强刺激皮肤画面
- 图上文字（on_image_copy）≤12 字，真实克制，不夸大、不医疗化
- 输出必须可被下游节点直接执行检索（必须提供 search_query）

【配图策略建议】
默认建议 4–6 张图：
1) 封面图（吸引点击）
2) 问题/踩坑图（共情）
3) 对比/流程图（提升收藏）
4) 关键步骤或要点图（解释）
5) 一屏总结卡（可截图保存）
（若正文不适合 6 张，可输出 3–4 张）

【输出要求（必须严格 JSON；不要输出多余文字）】
{
  "image_script": [
    {
      "img_id": "img_1",
      "purpose": "cover | empathy | step | compare | summary",
      "style": "realistic | minimal",
      "orientation": "vertical | square",
      "search_query": ["string", "string", "string"],
      "negative_constraints": ["string"],
      "on_image_copy": "string",
      "caption_hint": "string"
    }
  ],
  "global_notes": {
    "overall_style": "string",
    "avoid": ["string"],
    "recommended_count": number
  }
}

【质量要求】
- search_query 必须是可用于图库检索的短语（优先英文关键词 + 可选中文补充）
- negative_constraints 必须包含至少 3 条（例如 no logo/no watermark/no text/no face）
- caption_hint 是该图片在正文中出现时的配套一句话（不超过 25 字）
"""

In [130]:
messages10 = [
    SystemMessage(content=visual_prompt), 
    HumanMessage(content=f"这是final_title：{final_title}, 这是当前正文revised_md：{revised_md}, 请按 system 规则生成配图脚本。 ")
]
response10 = model.invoke(messages10)

In [133]:
final_title

'3种肤质保湿方案清单，照着做更安心'

In [136]:
image_script = json.loads(response10.content)

In [134]:
image_prompt = """
【角色】
你是“图片素材官（Image Sourcing）”。

【定位说明】
你不负责创意与策略，你只负责严格执行 Visual Director 给出的 image_script：
- 使用提供的 search_query 去图库 API 搜索
- 按 style/negative_constraints 筛选
- 输出可用图片清单与来源信息
如找不到合适图片，必须提出替代搜索词并再次尝试（在允许的次数内）。

【目标】
你将收到：
- image_script（数组，每项包含 purpose/style/search_query/negative_constraints 等）
请为每个 img_id 找到 2–4 张候选图（便于后续选择或 QA）。

【硬性约束（必须遵守）】
- 必须遵循 negative_constraints（如 no logo/no watermark/no face/no text）
- 不选含明显品牌露出、广告排版、水印、二维码、联系方式的图片
- 不选医疗诊疗画面或令人不适画面
- 图片应清晰、构图简洁、风格统一（实拍/简约）
- 输出必须包含来源与授权线索（provider/source_url/author等）
- 不要编造不存在的图片或链接；如果无结果必须如实说明并给出替代 query

【输出要求（必须严格 JSON；不要输出多余文字）】
{
  "image_candidates": [
    {
      "img_id": "string",
      "purpose": "string",
      "selected": [
        {
          "provider": "unsplash | pexels",
          "image_url": "string",
          "source_url": "string",
          "author": "string",
          "license_hint": "string",
          "why_match": "string"
        }
      ],
      "fallback_queries": ["string"]
    }
  ],
  "notes": ["string"]
}

【数量要求】
- 每个 img_id：selected 至少 2 张，最多 4 张
- 若不足 2 张：必须填写 fallback_queries，并说明原因
"""

In [ ]:
messages11 = [
    SystemMessage(content=image_prompt), 
    HumanMessage(content=f"这是image_script：{image_script}, 请按 system 规则生成搜索配图。 ")
]
response11 = model.invoke(messages11)

In [138]:
def research_plan_node(state: AgentState):
    queries = model.with_structured_output(Queries).invoke([
        SystemMessage(content=RESEARCH_PLAN_PROMPT),
        HumanMessage(content=state['task'])
    ])
    content = state['content'] or []
    for q in queries.queries:
        response = tavily.search(query=q, max_results=2)
        for r in response['results']:
            content.append(r['content'])
    return {"content": content}

NameError: name 'AgentState' is not defined

In [ ]:
def generation_node(state: AgentState):
    content = "\n\n".join(state['content'] or [])
    user_message = HumanMessage(
        content=f"{state['task']}\n\nHere is my plan:\n\n{state['plan']}")
    messages = [
        SystemMessage(
            content=WRITER_PROMPT.format(content=content)
        ),
        user_message
        ]
    response = model.invoke(messages)
    return {
        "draft": response.content, 
        "revision_number": state.get("revision_number", 1) + 1
    }


In [ ]:
def reflection_node(state: AgentState):
    messages = [
        SystemMessage(content=REFLECTION_PROMPT), 
        HumanMessage(content=state['draft'])
    ]
    response = model.invoke(messages)
    return {"critique": response.content}

In [ ]:
def research_critique_node(state: AgentState):
    queries = model.with_structured_output(Queries).invoke([
        SystemMessage(content=RESEARCH_CRITIQUE_PROMPT),
        HumanMessage(content=state['critique'])
    ])
    content = state['content'] or []
    for q in queries.queries:
        response = tavily.search(query=q, max_results=2)
        for r in response['results']:
            content.append(r['content'])
    return {"content": content}

In [ ]:
def should_continue(state):
    if state["revision_number"] > state["max_revisions"]:
        return END
    return "reflect"

In [ ]:
builder = StateGraph(AgentState)

In [ ]:
builder.add_node("planner", plan_node)
builder.add_node("generate", generation_node)
builder.add_node("reflect", reflection_node)
builder.add_node("research_plan", research_plan_node)
builder.add_node("research_critique", research_critique_node)

In [ ]:
builder.set_entry_point("planner")

In [ ]:
builder.add_conditional_edges(
    "generate", 
    should_continue, 
    {END: END, "reflect": "reflect"}
)


In [ ]:
builder.add_edge("planner", "research_plan")
builder.add_edge("research_plan", "generate")

builder.add_edge("reflect", "research_critique")
builder.add_edge("research_critique", "generate")

In [ ]:
graph = builder.compile(checkpointer=memory)

In [ ]:
from IPython.display import Image

Image(graph.get_graph().draw_png())

In [ ]:
thread = {"configurable": {"thread_id": "1"}}
for s in graph.stream({
    'task': "what is the difference between langchain and langsmith",
    "max_revisions": 2,
    "revision_number": 1,
}, thread):
    print(s)

## Essay Writer Interface

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from helper import ewriter, writer_gui

In [ ]:
MultiAgent = ewriter()
app = writer_gui(MultiAgent.graph)
app.launch()